# How it is asked: one question per criteria section

**One factor changed relative to [06_criteria_in_question.ipynb](06_criteria_in_question.ipynb)** (all full-text criteria in the instructions of one `include` question, `state` = record only): the criteria are split into four `noul` questions, one per section of *Criteria for considering studies for this review*. All four are asked in the same call. `state` still holds only the record (title, abstract); no review title, no research questions.

| question | section (word for word) |
|---|---|
| `studies` | Types of studies |
| `participants` | Types of participants |
| `interventions` | Types of interventions |
| `outcomes` | Types of outcome measures |

All 20 reviews have exactly these four sections and no text outside them, so the split needs no model and no labels.

**Combining the four probabilities, fixed before looking at results:** main score = product of all four; second view = minimum. Ablations from the same answers, no extra calls: leave one section out, and each section alone.

**Fixed on 2026-11-17, after the 10-review subset and before the other 10 reviews were run:** the subset suggested that the outcomes question hurts (it is the lowest probability for 98 of 129 content-level includes; leave-one-review-out weight tuning always chose outcome weight 0 or 0.25). Rule to test on the 10 reviews outside the subset: **product of studies, participants and interventions (outcomes left out)**. No further weights are tuned. The held-out evaluation is the last section of this notebook.

Labels: CLEF TAR abstract-level (`label_abs`) and content-level (`label_content`).

Nothing paid runs unless `RUN_SMOKE`, `RUN_SUBSET` or `RUN_PAID` is set to `True` below.

In [ ]:
import itertools, json, sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
import screening as S

RUN_SMOKE = False    # one paid call
RUN_SUBSET = False   # the 10 reviews with < 1000 records (3,637 records); see the cost estimate
RUN_PAID = True      # all 20 reviews; read the cost estimate first
OUT = S.out_dir("fulltext_criteria_split_questions")
COMPARE = {"baseline": S.out_dir("baseline"),
           "criteria_in_state": S.out_dir("review_fulltext_criteria_only"),
           "one_question": S.out_dir("fulltext_criteria_in_question")}

## Data and split

In [ ]:
records, CRITERIA = S.load_records()
records = S.add_bm25(records, CRITERIA)
FT = json.loads((ROOT / "data" / "processed" / "tar2019_intervention_fulltext_criteria.json").read_text())
assert set(FT) == set(CRITERIA)

SECTIONS = {"studies": "Types of studies", "participants": "Types of participants",
            "interventions": "Types of interventions", "outcomes": "Types of outcome measures"}

def split(criteria):
    """{question key: section text without its heading}, split at the '#### Types of ...' headings."""
    parts, key = {}, None
    for line in criteria.split("\n")[1:]:          # first line is the 'Criteria for considering ...' heading
        head = line.lstrip("#").strip()
        if line.startswith("#### ") and head in SECTIONS.values():
            key = next(k for k, v in SECTIONS.items() if v == head)
            parts[key] = []
        else:
            parts[key].append(line)
    return {k: "\n".join(v).strip() for k, v in parts.items()}

PARTS = {r: split(v["criteria"]) for r, v in FT.items()}
assert all(set(p) == set(SECTIONS) and all(p.values()) for p in PARTS.values()), "a review lacks a section"

n = records.groupby("review").size()
SUBSET = sorted(n[n < 1000].index)
print("subset:", len(SUBSET), "reviews,", int(n[SUBSET].sum()), "records")
pd.DataFrame({k: {r: len(p[k]) for r, p in PARTS.items()} for k in SECTIONS}).add_suffix("_chars")

## Decision model input

In [ ]:
def build_body(review, title, abstract):
    return {
        "model": S.MODEL,
        "state": {"record": {"title": title, "abstract": abstract}},
        "questions": {
            k: {"type": "noul",
                "instructions": f"The record meets this eligibility criterion of the systematic review ({SECTIONS[k].lower()}): "
                                f"it meets the stated requirements and none of the stated exclusions.\n\n{PARTS[review][k]}"}
            for k in SECTIONS
        },
    }

print(json.dumps(build_body(*records.loc[0, ["review", "title", "abstract"]])["questions"], indent=1))

## Cost estimate (no API call)

Same criteria text as the one-question run, plus a short sentence per question. Output tokens are free at the moment.

In [ ]:
est = S.estimate_cost(records, build_body)
print(f"subset ({len(SUBSET)} reviews): ${est.loc[SUBSET, 'usd'].sum():.2f}")

## Smoke test (one paid call)

Checks that the API answers four questions in one call, and compares tokens and cost with the one-question run for the same record.

In [ ]:
if RUN_SMOKE:
    rec = records.loc[0, ["review", "pmid", "title", "abstract"]]
    resp = S.call(build_body(rec.review, rec.title, rec.abstract))
    print(json.dumps(resp, indent=1))
    one = {r["pmid"]: r for r in map(json.loads, (COMPARE["one_question"] / f"{rec.review}.jsonl").open())}[rec.pmid]["response"]
    print("one question, same record:", one["usage"]["input_tokens"], "input tokens,", one["usage"]["output_tokens"], "output tokens,",
          one["usage"]["cost"], "USD, score", one["answers"]["include"]["noul"])

## Subset (paid)

In [ ]:
if RUN_SUBSET:
    S.run_reviews(records, SUBSET, build_body, OUT)

## Full run (paid)

In [ ]:
if RUN_PAID:
    S.run_reviews(records, CRITERIA, build_body, OUT)

## Results

All configurations on the same records: the reviews with a cache file for this run. A failed call, or a missing answer to any of the four questions, gives that record the lowest score for every combination (counted below).

In [ ]:
done = sorted(p.stem for p in OUT.glob("*.jsonl"))
print(len(done), "of", len(CRITERIA), "reviews run")
sub = records[records.review.isin(done)].copy()
for col, path in COMPARE.items():
    sub = S.add_scores(sub, path, col)

rows = []
for r in done:
    for row in map(json.loads, (OUT / f"{r}.jsonl").open()):
        ans = row["response"].get("answers", {})
        rows.append({"review": r, "pmid": row["pmid"], **{k: ans.get(k, {}).get("noul") for k in SECTIONS}})
P = pd.DataFrame(rows, columns=["review", "pmid", *SECTIONS]).drop_duplicates(["review", "pmid"], keep="last")
sub = sub.merge(P, on=["review", "pmid"], how="left")
incomplete = sub[list(SECTIONS)].isna().any(axis=1)
print("records with a failed call or a missing answer (ranked last):", int(incomplete.sum()))

def combine(keys, how):
    v = sub[list(keys)]
    s = v.prod(axis=1) if how == "product" else v.min(axis=1)
    return s.where(~incomplete, -1.0)

COMBOS = {"split: product (all 4)": (tuple(SECTIONS), "product"), "split: min (all 4)": (tuple(SECTIONS), "min")}
for k in SECTIONS:
    rest = tuple(x for x in SECTIONS if x != k)
    COMBOS[f"ablation: product without {k}"] = (rest, "product")
for k in SECTIONS:
    COMBOS[f"ablation: {k} only"] = ((k,), "product")
for name, (keys, how) in COMBOS.items():
    sub[name] = combine(keys, how)

results = {c: S.evaluate(sub, c) for c in ("bm25", *COMPARE, *COMBOS)}
main = pd.concat({c: results[c] for c in ("baseline", "criteria_in_state", "one_question", "split: product (all 4)", "split: min (all 4)")})
main

In [ ]:
# ablations: MAP and WSS@95% per label level
abl = pd.DataFrame({c: {(lab, m): results[c].loc[lab, m] for lab in ("label_abs", "label_content") for m in ("MAP", "WSS@95%", "TNR@95%")}
                    for c in ("one_question", *COMBOS)}).T if done else pd.DataFrame()
abl

In [ ]:
# usage: tokens and cost for exactly these records
def usage(path, reviews):
    tok = out_tok = cost = n = 0
    for r in reviews:
        for row in map(json.loads, (path / f"{r}.jsonl").open()):
            u = row["response"].get("usage", {})
            tok += u.get("input_tokens", 0); out_tok += u.get("output_tokens", 0); cost += u.get("cost") or 0; n += 1
    return {"calls": n, "input_tokens": tok, "output_tokens": out_tok, "usd": round(cost, 3), "tokens_per_call": round(tok / max(n, 1))}

pd.DataFrame({c: usage(p, done) for c, p in {**COMPARE, "split": OUT}.items()}).T

In [ ]:
# which section is the bottleneck: for each record, the question with the lowest probability
if done:
    low = sub.loc[~incomplete, list(SECTIONS)].idxmin(axis=1)
    print(pd.crosstab(low, [sub.loc[~incomplete, "label_abs"], sub.loc[~incomplete, "label_content"]], margins=True))
    print()
    print("median probability per section, by abstract-level label:")
    print(sub[~incomplete].groupby("label_abs")[list(SECTIONS)].median().round(3))

In [ ]:
# per-review MAP, main score vs one question
per = {}
for r, df in sub.groupby("review"):
    for label in ("label_abs", "label_content"):
        for c in ("baseline", "one_question", "split: product (all 4)"):
            m = S.metrics(df.sort_values([c, "bm25", "pmid"], ascending=[False, False, True])[label])
            per.setdefault(r, {})[(label, c)] = round(m["MAP"] * 100, 1) if m else None
per = pd.DataFrame(per).T
for label in ("label_abs", "label_content") if len(per) else ():
    d = per[(label, "split: product (all 4)")] - per[(label, "one_question")]
    print(f"{label}: split (product) vs one question, better {(d > 0).sum()} / worse {(d < 0).sum()} of {d.notna().sum()} reviews")
per

## Held-out test: the 10 reviews outside the subset

The rule fixed before these reviews were run (product without outcomes) against the equal-weight product, one question with all criteria, and the baseline. Only reviews outside `SUBSET`.

In [ ]:
HELD_OUT = [r for r in done if r not in SUBSET]
print(len(HELD_OUT), "held-out reviews:", HELD_OUT)
if HELD_OUT:
    ho = sub[sub.review.isin(HELD_OUT)]
    cols = ["baseline", "one_question", "split: product (all 4)", "ablation: product without outcomes"]
    display(pd.concat({c: S.evaluate(ho, c) for c in cols}))
    per_ho = {}
    for r, df in ho.groupby("review"):
        for label in ("label_abs", "label_content"):
            for c in cols:
                m = S.metrics(df.sort_values([c, "bm25", "pmid"], ascending=[False, False, True])[label])
                per_ho.setdefault(r, {})[(label, c)] = round(m["MAP"] * 100, 1) if m else None
    per_ho = pd.DataFrame(per_ho).T
    for label in ("label_abs", "label_content"):
        for ref in ("one_question", "split: product (all 4)"):
            d = per_ho[(label, "ablation: product without outcomes")] - per_ho[(label, ref)]
            print(f"{label}: without outcomes vs {ref}: better {(d > 0).sum()} / worse {(d < 0).sum()} of {d.notna().sum()} reviews")
    display(per_ho)